# Day 2: XGBoost Regressor from Scratch

In this notebook we build an XGBoost regressor, extending the classifier. The core boosting idea is identical — trees are built sequentially, each correcting the mistakes of the previous ones using gradients and hessians from a Taylor-approximated loss. What changes is the loss function, and that single change ripples into the gradient, the hessian, and how predictions are updated. There is no log-odds space here, no sigmoid — the model works directly in the output space throughout.

In [5]:
from wrapped_models.xg_boost_regression_tree import XG_Boost_Regressor_Tree
import numpy as np
import pandas as pd

## No Log-Odds, No Sigmoid

In the classifier, predictions lived in **log-odds space**. The model accumulated corrections to $z$, and only converted to probability via sigmoid at the very end. This was necessary because probabilities are bounded between 0 and 1 — you cannot add tree outputs directly to them.

In regression the target is an unbounded real number, so there is no need for any transformation. The model predicts directly:

$$\hat{y} = \hat{y}_0 + \sum_{t=1}^{T} \eta \cdot f_t(x)$$

Each tree output is added directly to the running prediction. No sigmoid, no log-odds — the update rule is the same but the space is simpler.

The initial prediction $\hat{y}_0$ is the **mean of the training targets** — not 0 or 0.5. This is the best constant prediction before any tree is built. The same reasoning applies to the classifier — starting from the class mean rather than 0 log-odds is strictly better, and real XGBoost does exactly that.

In [6]:
def compute_baseline_prediction(y: np.ndarray) -> float:
    return np.mean(y)

## MSE Loss and the Taylor Approximation

The classifier minimized log-loss. Here we minimize **Mean Squared Error**:

$$L = (y - \hat{y})^2$$

The Taylor approximation is identical in form to the classifier:

$$L(\hat{y} + \Delta) \approx L(\hat{y}) + g \cdot \Delta + \frac{1}{2} h \cdot \Delta^2$$

where $\Delta$ is the output of the new tree — the correction being added to the current prediction. XGBoost still finds the optimal $\Delta$ analytically by minimizing this quadratic, using the vertex formula. What changes is what $g$ and $h$ are, because the loss function is different.

## Gradients and Hessians

Taking derivatives of $L = (y - \hat{y})^2$ with respect to $\hat{y}$:

$$g = \frac{\partial L}{\partial \hat{y}} = 2(\hat{y} - y)$$

$$h = \frac{\partial^2 L}{\partial \hat{y}^2} = 2$$

The hessian is **constant** — the same value for every sample regardless of the current prediction. This is the fundamental difference from the classifier, where $h = p(1-p)$ varied per sample based on how confident the model was.

In the classifier, the hessian carried real information. Uncertain predictions near $p = 0.5$ produced large hessians, which down-weighted the leaf correction for those samples — the model was cautious about overcommitting on uncertain predictions. Confident predictions near 0 or 1 produced small hessians and had little influence on the leaf output. This gave the classifier **adaptive per-sample step sizes**.

In regression $h = 2$ for everyone. Every sample is weighted equally. The hessian adds nothing beyond sample count, and it cancels with the 2 in the gradient when computing the similarity score:

$$SS = \frac{\left(\sum g_i\right)^2}{N + \lambda}$$

instead of the classifier's $\frac{\left(\sum g_i\right)^2}{\sum h_i + \lambda}$. This is why the regressor tree does not need hessians passed in at all.

In [7]:
def compute_gradients(y_pred: np.ndarray, y: np.ndarray) -> np.ndarray:
    # we skip *2 for simplicity - since it cancels out with the 2 from hessian
    return y_pred - y

## Leaf Value — Why It Simplifies to Mean Residual

The optimal leaf output is found by minimizing the Taylor-approximated loss with respect to $\Delta$. Taking the derivative and setting it to zero gives the vertex formula $-b / 2a$:

$$w^* = -\frac{\sum g_i}{\sum h_i + \lambda}$$

Substituting $g_i = 2(\hat{y}_i - y_i)$ and $h_i = 2$:

$$w^* = -\frac{\sum 2(\hat{y}_i - y_i)}{2N + \lambda}$$

As $\lambda \to 0$ the 2s cancel and this becomes:

$$w^* = -\frac{\sum (\hat{y}_i - y_i)}{N} = -\overline{\text{residual}}$$

The leaf outputs the **negative mean residual** of all samples that land in it. If the model is over-predicting on average in that leaf ($\hat{y} > y$), the gradient is positive, $w^*$ is negative, and the correction pulls predictions down. If under-predicting, the opposite happens. The sign always corrects in the right direction.

Compare this to the classifier leaf:

$$w^*_{clf} = -\frac{\sum g_i}{\sum h_i + \lambda} = -\frac{\sum (p_i - y_i)}{\sum p_i(1-p_i) + \lambda}$$

The structure is identical — the difference is that the denominator in the classifier is a weighted sum that varies with model confidence, while in the regressor it is just $N$.

## What the Tree Learns to Group

In the classifier, the tree searched for splits that best separated samples by class — grouping likely dead from likely alive, using both gradients and hessians to evaluate split quality.

In regression there are no classes. Instead the tree searches for splits that group samples whose **residuals point in the same direction**. The Gain formula rewards this:

$$Gain = SS_{left} + SS_{right} - SS_{parent} - \gamma$$

The similarity score $SS = \frac{(\sum g_i)^2}{N + \lambda}$ is large when residuals inside a node do not cancel each other out — they mostly have the same sign. A split is good when the parent mixes positive and negative residuals (they cancel, small $SS$) and after splitting each child contains residuals that consistently point one way (large $SS$ each).

A concrete example: suppose after tree 1 the model under-predicts expensive houses and over-predicts cheap ones. Tree 2 finds that splitting on `price > threshold` perfectly separates these two residual groups — negative residuals on one side, positive on the other. The gain is large, the split is accepted, and each leaf applies a meaningful correction. Without the split, the mixed residuals would average toward zero and the leaf would correct nothing.

This is the same logic as the classifier — find groups where the current model is consistently wrong in the same direction, then correct them. The difference is only what "wrong in the same direction" means: in classification it is misclassified samples, in regression it is residuals sharing the same sign.

## The Boosting Loop

The regressor loop mirrors the classifier exactly, with two differences: no sigmoid conversion anywhere, and the initial prediction is $\bar{y}$ not 0.

At each iteration:

1. Compute gradients $g_i = \hat{y}_i - y_i$ from the current predictions
2. Build a tree on those gradients
3. Update predictions:

$$\hat{y}^{(t)} = \hat{y}^{(t-1)} + \eta \cdot f_t(x)$$

There is no probability conversion at any step — gradients are computed directly from $\hat{y} - y$ in real space. The classifier had to convert log-odds to probabilities before computing gradients each round because its gradient formula required $p$, not $z$. Here $\hat{y}$ is already in the right space.

In [8]:
class XGBoostRegressor:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3,
                 min_samples_split=2, reg_lambda=1.0, gamma=0.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.reg_lambda = reg_lambda
        self.gamma = gamma
        self.trees = []
        self.y_mean = None

    def fit(self, X: pd.DataFrame, y: np.ndarray) -> None:
        # start with the baseline for every sample
        self.y_mean = compute_baseline_prediction(y=y)

        predictions = np.full(shape=y.shape, fill_value=self.y_mean)

        for _ in range(self.n_estimators):
            gradients = compute_gradients(y_pred=predictions, y=y)

            new_tree = XG_Boost_Regressor_Tree(
                max_depth=self.max_depth,
                min_samples=self.min_samples_split,
                lam=self.reg_lambda,
                gamma=self.gamma
            )
            
            new_tree.fit(X=X, gradients=gradients)  # build a new tree, find best splits

            self.trees.append(new_tree)

            # update predictions based on the new residuals
            residuals = np.array(new_tree.predict(X))
            predictions = predictions + self.learning_rate * residuals


    def predict(self, X: pd.DataFrame) -> np.ndarray:
        # start from initial baseline
        predictions = np.full(shape=X.shape[0], fill_value=self.y_mean)
        
        for tree in self.trees:
            # predict() returns list of floats, not np.array, so convert it 
            predictions += self.learning_rate * np.array(tree.predict(X))
        
        return predictions

## Evaluation

We evaluate using standard regression metrics:

- **MSE** — mean squared error, penalizes large errors more heavily due to squaring
- **RMSE** — square root of MSE, error in the same units as the target, easier to interpret
- **MAE** — mean absolute error, treats all errors equally regardless of size
- **R²** — proportion of variance in the target explained by the model, 1.0 is perfect

Since the dataset is linear with low noise and we know the true weights, R² should be very close to 1.0 — confirming the boosting procedure is working correctly.

In [9]:
# create dummy dataset
np.random.seed(42)
n_samples = 100
X = pd.DataFrame({
    'feature_1': np.random.randn(n_samples),
    'feature_2': np.random.randn(n_samples),
    'feature_3': np.random.randn(n_samples),
})

true_weights = np.array([2.0, -1.5, 3.0])
true_bias = 4.0
y = X.values @ true_weights + true_bias + np.random.randn(n_samples) * 0.5

split = int(0.8 * n_samples)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y[:split], y[split:]

reg = XGBoostRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    min_samples_split=2,
    reg_lambda=1.0,
    gamma=0.0
)
reg.fit(X_train, y_train)

y_pred = np.array(reg.predict(X_test))

mse  = np.mean((y_test - y_pred) ** 2)
rmse = np.sqrt(mse)
mae  = np.mean(np.abs(y_test - y_pred))
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
r2   = 1 - ss_res / ss_tot

print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

MSE:  0.9077
RMSE: 0.9527
MAE:  0.7918
R²:   0.9428
